**Why does this matter in Quant Finance?**

Imagine you have a (252, 4) matrix of daily returns for 4 stocks, and you want to subtract each stock's mean return from every single day. The naive approach is a loop — iterating over 252 rows one by one. Broadcasting lets NumPy do it in one line, no loop, running at C speed under the hood.

This is the difference between a backtest that runs in 0.2 seconds vs 20 seconds on real data. At scale, it's not even close. 🚀    

**Part 1 - Vectorisation**

Vectorisation means applying an operation to an entire array at once rather than element by element in a Python loop

In [5]:
import numpy as np

In [13]:
returns = np.array([0.01, -0.02, 0.015, -0.03, 0.025])

# Non-vectorised — slow Python loop
result = []
for r in returns:
    result.append(r * 100)

# Vectorised — fast, clean, Pythonic
result = returns * 100

#Under the hood, NumPy pushes that operation down into compiled C code — orders of magnitude faster than a Python loop.


**Part 2 - Broadcasting**

Broadcasting is Numpy's ruleset for how to handle operations between arrays of different shapes - without copying data or writing loop. This is like how we covered in module 4.

In [14]:
#A simple example of scalar broadcasting is multiplying an array by a single number. The number is "stretched" across all elements of the array, and the operation is applied element-wise.

returns = np.array([0.01, -0.02, 0.03, -0.01, 0.02])

# Scalar broadcast — the 100 is "stretched" across all elements
returns * 100
# [ 1.  -2.   3.  -1.   2.]

array([ 1., -2.,  3., -1.,  2.])

In [15]:
# 3 days × 4 stocks return matrix
returns = np.array([[ 0.01, -0.02,  0.03,  0.00],
                    [-0.01,  0.02, -0.01,  0.03],
                    [ 0.02,  0.01,  0.00, -0.02]])
# shape: (3, 4)

# Mean return per stock (one value per column)
mean_returns = np.array([0.01, 0.00, 0.01, 0.00])
# shape: (4,)

# Subtract mean from every day — broadcasting handles the shape mismatch!
excess_returns = returns - mean_returns
print(excess_returns)
# [[ 0.    -0.02   0.02   0.  ]
#  [-0.02   0.02  -0.02   0.03]
#  [ 0.01   0.01  -0.01  -0.02]]


[[ 0.   -0.02  0.02  0.  ]
 [-0.02  0.02 -0.02  0.03]
 [ 0.01  0.01 -0.01 -0.02]]


NumPy sees (3, 4) - (4,) and automatically stretches the (4,) array across all 3 rows. No loop needed. 🎯

Broadcasting visualised

```
returns       mean_returns     excess_returns
(3, 4)    -   (4,)         =   (3, 4)

[0.01  -0.02  0.03  0.00]     [0.01  0.00  0.01  0.00]
[-0.01  0.02 -0.01  0.03]  -  [0.01  0.00  0.01  0.00]  =  result
[0.02   0.01  0.00 -0.02]     [0.01  0.00  0.01  0.00]
                ↑
        mean_returns gets
        "stretched" across
        all 3 rows automatically

**📏 Part 3 — The Broadcasting Rules**

NumPy follows strict rules to decide if two shapes are compatible:

Rule: Compare shapes from the right. Dimensions are compatible if they are either equal or one of them is 1.

````
(3, 4) and (4,)   → compatible ✅  (4 == 4)
(3, 4) and (3,)   → incompatible ❌ (4 ≠ 3)
(3, 4) and (1, 4) → compatible ✅  (3 vs 1 → stretch)
(3, 4) and (3, 1) → compatible ✅  (4 vs 1 → stretch)

**⚡ Part 4 — Common Vectorised Operations**

In [16]:
returns = np.array([0.01, -0.02, 0.03, -0.01, 0.02])

# Arithmetic — all vectorised
returns + 0.005      # add risk-free rate adjustment
returns * 100        # convert to percentages
returns ** 2         # squared returns (used in volatility calcs)

# NumPy universal functions (ufuncs) — also vectorised
np.abs(returns)      # absolute returns
np.sqrt(np.abs(returns))   # square root
np.exp(returns)      # exponential (used in log-return calculations)
np.log(1 + returns)  # log returns — extremely common in quant finance!

array([ 0.00995033, -0.02020271,  0.0295588 , -0.01005034,  0.01980263])

💡 np.log(1 + returns) is one of the most used operations in quant finance — log returns have nice mathematical properties (they're additive across time, and more normally distributed than simple returns).

**Tasks**

Task 1: Calculate the mean return per stock (hint: np.mean with axis argument)<br>
Task 2: Subtract the mean from every day's return using broadcasting<br>
Task 3: Convert all simple returns to log returns using np.log(1 + returns)<br>
Task 4: Calculate the total log return for each stock (sum across days)<br>

In [17]:
# Daily simple returns for 3 stocks over 5 days
returns = np.array([[ 0.01,  0.02, -0.01],
                    [-0.02,  0.01,  0.03],
                    [ 0.03, -0.01,  0.02],
                    [ 0.00,  0.03, -0.02],
                    [ 0.02,  0.01,  0.01]])
# shape: (5, 3)

In [18]:
#Task 1
# Calculate the mean return for each stock (column-wise mean)
mean_returns = np.mean(returns, axis=0)
print("Mean returns for each stock:", mean_returns)

#Task 2
#Subtract the mean from every day's returns using broadcasting
print(returns - mean_returns,"\n")

#Task 3
#Convert all simple returns to log returns using np.log(1+ returns)
print(np.log(1 + returns),"\n")

#Task 4
#Calculate the total log return for each stock (sum across days)
log_returns = np.log(np.sum(returns, axis=0))
print(log_returns)

Mean returns for each stock: [0.008 0.012 0.006]
[[ 0.002  0.008 -0.016]
 [-0.028 -0.002  0.024]
 [ 0.022 -0.022  0.014]
 [-0.008  0.018 -0.026]
 [ 0.012 -0.002  0.004]] 

[[ 0.00995033  0.01980263 -0.01005034]
 [-0.02020271  0.00995033  0.0295588 ]
 [ 0.0295588  -0.01005034  0.01980263]
 [ 0.          0.0295588  -0.02020271]
 [ 0.01980263  0.00995033  0.00995033]] 

[-3.21887582 -2.81341072 -3.5065579 ]
